# DNN for `CO2_purity` and `CO2_recovery`

이 노트북은 지정한 15개 입력 변수만 사용해서 `CO2_purity`, `CO2_recovery`를 각각 별도 DNN으로 학습합니다. 데이터는 900개를 `train 720 / validation 90 / test 90`으로 분할하고, 작은 데이터셋을 고려해 과적합 방지용 정규화를 포함합니다.

In [3]:
import os
import random
import warnings
from pathlib import Path

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers, regularizers

warnings.filterwarnings('ignore', message='Protobuf gencode version .*')

plt.style.use('seaborn-v0_8-whitegrid')

In [11]:
SEED = 42
FEATURE_COLS = [
    'tADS', 'PL', 'v0', 't_press', 't_depress',
    'h2o_ppmv', 'sox_ppmv', 'nox_ppmv', 'dust_mg_Nm3',
    'co2_mol_frac', 'capacity_factor', 'affinity_factor_co2',
    'mtc_factor', 'dax_factor', 'deactivation_index'
]
TARGET_COLS = ['CO2_purity', 'CO2_recovery']

base_dir = Path.cwd()

if (base_dir / 'dataset' / 'data.csv').exists():
    mof_dir = base_dir
elif (base_dir / 'mof' / 'dataset' / 'data.csv').exists():
    mof_dir = base_dir / 'mof'

data_path = mof_dir / 'dataset' / 'data.csv'
model_dir = mof_dir / 'model'

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)

print(f'Data path: {data_path}')
print(f'Model output dir: {model_dir}')

NameError: name 'mof_dir' is not defined

In [6]:
df = pd.read_csv(data_path).copy()

if 'valid_physics' in df.columns:
    df = df[df['valid_physics'].fillna(False)].copy()
if 'status' in df.columns:
    df = df[df['status'].eq('ok')].copy()

required_cols = FEATURE_COLS + TARGET_COLS
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f'Missing required columns: {missing_cols}')

df = df.dropna(subset=required_cols).reset_index(drop=True)
if len(df) != 900:
    raise ValueError(f'Expected 900 usable rows, found {len(df)} rows.')

train_df, temp_df = train_test_split(df, test_size=180, random_state=SEED, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=90, random_state=SEED, shuffle=True)

split_summary = pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'n_rows': [len(train_df), len(val_df), len(test_df)]
})

display(split_summary)
display(df[FEATURE_COLS + TARGET_COLS].head())

FileNotFoundError: [Errno 2] No such file or directory: 'mof\\dataset\\data.csv'

In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2': r2_score(y_true, y_pred),
    }


def build_model(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        layers.Dropout(0.15),
        layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        layers.Dropout(0.10),
        layers.Dense(16, activation='relu'),
        layers.Dense(1)
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss=keras.losses.Huber(),
        metrics=[keras.metrics.MeanAbsoluteError(name='mae')]
    )
    return model


def train_single_target(target_name):
    x_train = train_df[FEATURE_COLS].to_numpy(dtype=np.float32)
    x_val = val_df[FEATURE_COLS].to_numpy(dtype=np.float32)
    x_test = test_df[FEATURE_COLS].to_numpy(dtype=np.float32)

    y_train = train_df[target_name].to_numpy(dtype=np.float32).reshape(-1, 1)
    y_val = val_df[target_name].to_numpy(dtype=np.float32).reshape(-1, 1)
    y_test = test_df[target_name].to_numpy(dtype=np.float32).reshape(-1, 1)

    x_scaler = StandardScaler()
    y_scaler = StandardScaler()

    x_train_scaled = x_scaler.fit_transform(x_train).astype(np.float32)
    x_val_scaled = x_scaler.transform(x_val).astype(np.float32)
    x_test_scaled = x_scaler.transform(x_test).astype(np.float32)

    y_train_scaled = y_scaler.fit_transform(y_train).astype(np.float32)
    y_val_scaled = y_scaler.transform(y_val).astype(np.float32)

    model = build_model(input_dim=x_train_scaled.shape[1])
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=40,
            restore_best_weights=True,
            verbose=0
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=15,
            min_lr=1e-5,
            verbose=0
        )
    ]

    history = model.fit(
        x_train_scaled,
        y_train_scaled,
        validation_data=(x_val_scaled, y_val_scaled),
        epochs=500,
        batch_size=32,
        verbose=0,
        callbacks=callbacks
    )

    y_pred_scaled = model.predict(x_test_scaled, verbose=0)
    y_pred = y_scaler.inverse_transform(y_pred_scaled).ravel()
    y_true = y_test.ravel()

    metrics = regression_metrics(y_true, y_pred)
    metrics['best_epoch'] = int(np.argmin(history.history['val_loss']) + 1)
    metrics['epochs_ran'] = len(history.history['loss'])

    results_df = test_df[['sample_id']].copy() if 'sample_id' in test_df.columns else pd.DataFrame(index=test_df.index)
    results_df[target_name] = y_true
    results_df[f'pred_{target_name}'] = y_pred
    results_df[f'error_{target_name}'] = y_true - y_pred

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history['loss'], label='train')
    axes[0].plot(history.history['val_loss'], label='validation')
    axes[0].set_title(f'{target_name} loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Huber loss')
    axes[0].legend()

    axes[1].plot(history.history['mae'], label='train')
    axes[1].plot(history.history['val_mae'], label='validation')
    axes[1].set_title(f'{target_name} MAE')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('MAE')
    axes[1].legend()
    plt.tight_layout()
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    low = min(y_true.min(), y_pred.min())
    high = max(y_true.max(), y_pred.max())
    axes[0].scatter(y_true, y_pred, alpha=0.85)
    axes[0].plot([low, high], [low, high], linestyle='--', color='red')
    axes[0].set_title(f'{target_name}: actual vs predicted')
    axes[0].set_xlabel('Actual')
    axes[0].set_ylabel('Predicted')

    sample_axis = results_df['sample_id'] if 'sample_id' in results_df.columns else np.arange(len(results_df))
    axes[1].plot(sample_axis, y_true, marker='o', label='Actual')
    axes[1].plot(sample_axis, y_pred, marker='s', label='Predicted')
    axes[1].set_title(f'{target_name}: test samples')
    axes[1].set_xlabel('Sample')
    axes[1].set_ylabel(target_name)
    axes[1].legend()
    plt.tight_layout()
    plt.show()

    model.save(model_dir / f'{target_name}_dnn.keras')
    return model, results_df, metrics


In [ ]:
all_metrics = []
all_results = {}
all_models = {}

for target_name in TARGET_COLS:
    print(f'===== {target_name} =====')
    model, results_df, metrics = train_single_target(target_name)
    all_models[target_name] = model
    all_results[target_name] = results_df
    all_metrics.append({'target': target_name, **metrics})
    display(results_df.head())

metrics_df = pd.DataFrame(all_metrics)
display(metrics_df)

print('\nSaved model files:')
for target_name in TARGET_COLS:
    print(model_dir / f'{target_name}_dnn.keras')